<a href="https://colab.research.google.com/github/lucazini03/DeepLearning/blob/main/Infix_to_postfix_notation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
import math
from torch.utils.data import Dataset, DataLoader
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
# -------------------- Constants --------------------
OPERATORS = ['+', '-', '*', '/']
IDENTIFIERS = list('abcde')
SPECIAL_TOKENS = ['PAD', 'SOS', 'EOS']
SYMBOLS = ['(', ')', '+', '-', '*', '/']
VOCAB = SPECIAL_TOKENS + SYMBOLS + IDENTIFIERS + ['JUNK']

token_to_id = {tok: i for i, tok in enumerate(VOCAB)}
id_to_token = {i: tok for tok, i in token_to_id.items()}
VOCAB_SIZE = len(VOCAB)
PAD_ID = token_to_id['PAD']
EOS_ID = token_to_id['EOS']
SOS_ID = token_to_id['SOS']

MAX_DEPTH = 3
MAX_LEN = 4*2**MAX_DEPTH -2

The following cell contains the implementation of the neural network architecture we are going to use for this task: a **transformer**. I've followed linearly the approach presented by the original paper.

In [10]:
# -------------------- Transformer Implementation --------------------
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"


        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        attn_probs = torch.softmax(attn_scores, dim=-1)
        output = torch.matmul(attn_probs, V)
        return output

    def split_heads(self, x):
        batch_size, seq_length, d_model = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        batch_size, _, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)

    def forward(self, Q, K, V, mask=None):
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))

        attn_output = self.scaled_dot_product_attention(Q, K, V, mask)
        output = self.W_o(self.combine_heads(attn_output))
        return output

class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PositionWiseFeedForward, self).__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length):
        super(PositionalEncoding, self).__init__()

        pe = torch.zeros(max_seq_length, d_model)
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        attn_output = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        return x

class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(DecoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_output, src_mask, tgt_mask):
        attn_output = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(attn_output))
        attn_output = self.cross_attn(x, enc_output, enc_output, src_mask)
        x = self.norm2(x + self.dropout(attn_output))
        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout(ff_output))
        return x

class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout):
        super(Transformer, self).__init__()
        self.encoder_embedding = nn.Embedding(src_vocab_size, d_model)
        self.decoder_embedding = nn.Embedding(tgt_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_length)

        self.encoder_layers = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.decoder_layers = nn.ModuleList([DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])

        self.fc = nn.Linear(d_model, tgt_vocab_size)
        self.dropout = nn.Dropout(dropout)

    def generate_mask(self, src, tgt):
        src_mask = (src != PAD_ID).unsqueeze(1).unsqueeze(2)
        tgt_mask = (tgt != PAD_ID).unsqueeze(1).unsqueeze(3)
        seq_length = tgt.size(1)
        nopeak_mask = (1 - torch.triu(torch.ones(1, seq_length, seq_length), diagonal=1)).bool()
        tgt_mask = tgt_mask & nopeak_mask
        return src_mask, tgt_mask

    def forward(self, src, tgt):
        src_mask, tgt_mask = self.generate_mask(src, tgt)
        src_embedded = self.dropout(self.positional_encoding(self.encoder_embedding(src)))
        tgt_embedded = self.dropout(self.positional_encoding(self.decoder_embedding(tgt)))

        enc_output = src_embedded
        for enc_layer in self.encoder_layers:
            enc_output = enc_layer(enc_output, src_mask)

        dec_output = tgt_embedded
        for dec_layer in self.decoder_layers:
            dec_output = dec_layer(dec_output, enc_output, src_mask, tgt_mask)

        output = self.fc(dec_output)
        return output

In [11]:
# -------------------- Dataset Generation --------------------
def generate_infix_expression(max_depth):
    if max_depth == 0:
        return random.choice(IDENTIFIERS)
    elif random.random() < 0.5:
        return generate_infix_expression(max_depth - 1)
    else:
        left = generate_infix_expression(max_depth - 1)
        right = generate_infix_expression(max_depth - 1)
        op = random.choice(OPERATORS)
        return f'({left} {op} {right})'

def tokenize(expr):
    return [c for c in expr if c in token_to_id]

def infix_to_postfix(tokens):
    precedence = {'+': 1, '-': 1, '*': 2, '/': 2}
    output, stack = [], []
    for token in tokens:
        if token in IDENTIFIERS:
            output.append(token)
        elif token in OPERATORS:
            while stack and stack[-1] in OPERATORS and precedence[stack[-1]] >= precedence[token]:
                output.append(stack.pop())
            stack.append(token)
        elif token == '(':
            stack.append(token)
        elif token == ')':
            while stack and stack[-1] != '(':
                output.append(stack.pop())
            stack.pop()
    while stack:
        output.append(stack.pop())
    return output

def encode(tokens, max_len=MAX_LEN):
    ids = [token_to_id[t] for t in tokens] + [EOS_ID]
    return ids + [PAD_ID] * (max_len - len(ids))

def decode_sequence(token_ids, id_to_token, pad_token='PAD', eos_token='EOS'):
    tokens = []
    for token_id in token_ids:
        token = id_to_token.get(token_id, '?')
        if token == eos_token:
            break
        if token != pad_token:
            tokens.append(token)
    return ' '.join(tokens)

def shift_right(seqs):
    shifted = np.zeros_like(seqs)
    shifted[:, 1:] = seqs[:, :-1]
    shifted[:, 0] = SOS_ID
    return shifted

class InfixPostfixDataset(Dataset):
    def __init__(self, num_samples):
        self.X, self.Y = self.generate_dataset(num_samples)
        self.decoder_input = shift_right(self.Y)

    def generate_dataset(self, n):
        X, Y = [], []
        for _ in range(n):
            expr = generate_infix_expression(MAX_DEPTH)
            infix = tokenize(expr)
            postfix = infix_to_postfix(infix)
            X.append(encode(infix))
            Y.append(encode(postfix))
        return np.array(X), np.array(Y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return {
            'encoder_input': torch.tensor(self.X[idx], dtype=torch.long),
            'decoder_input': torch.tensor(self.decoder_input[idx], dtype=torch.long),
            'target': torch.tensor(self.Y[idx], dtype=torch.long)
        }

In [12]:
# -------------------- Training Setup --------------------
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Model configuration
d_model = 128
num_heads = 8
num_layers = 3
d_ff = 512
dropout = 0.1

model = Transformer(VOCAB_SIZE, VOCAB_SIZE, d_model, num_heads, num_layers, d_ff, MAX_LEN, dropout)
print(f"Model has {count_parameters(model):,} trainable parameters")

Model has 1,394,319 trainable parameters


In [13]:
# Create datasets
train_dataset = InfixPostfixDataset(10000)
val_dataset = InfixPostfixDataset(1000)

# Data loaders
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

# Loss and optimizer
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
optimizer = optim.Adam(model.parameters(), lr=0.0001, betas=(0.9, 0.98), eps=1e-9)

In [14]:
# -------------------- Training Loop --------------------
def train_epoch(model, dataloader, criterion, optimizer):
    model.train()
    total_loss = 0
    for batch in dataloader:
        optimizer.zero_grad()
        encoder_input = batch['encoder_input']
        decoder_input = batch['decoder_input']
        target = batch['target']

        output = model(encoder_input, decoder_input)
        loss = criterion(output.view(-1, VOCAB_SIZE), target.view(-1))
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(dataloader)

def evaluate(model, dataloader, criterion):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in dataloader:
            encoder_input = batch['encoder_input']
            decoder_input = batch['decoder_input']
            target = batch['target']

            output = model(encoder_input, decoder_input)
            loss = criterion(output.view(-1, VOCAB_SIZE), target.view(-1))
            total_loss += loss.item()
    return total_loss / len(dataloader)

# Training
num_epochs = 20
for epoch in range(num_epochs):
    train_loss = train_epoch(model, train_loader, criterion, optimizer)
    val_loss = evaluate(model, val_loader, criterion)
    print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f} - Val Loss: {val_loss:.4f}")

Epoch 1/20 - Train Loss: 1.5816 - Val Loss: 0.9288
Epoch 2/20 - Train Loss: 0.7413 - Val Loss: 0.4201
Epoch 3/20 - Train Loss: 0.4541 - Val Loss: 0.2375
Epoch 4/20 - Train Loss: 0.3144 - Val Loss: 0.1518
Epoch 5/20 - Train Loss: 0.2263 - Val Loss: 0.1041
Epoch 6/20 - Train Loss: 0.1696 - Val Loss: 0.0723
Epoch 7/20 - Train Loss: 0.1309 - Val Loss: 0.0492
Epoch 8/20 - Train Loss: 0.1024 - Val Loss: 0.0441
Epoch 9/20 - Train Loss: 0.0822 - Val Loss: 0.0365
Epoch 10/20 - Train Loss: 0.0691 - Val Loss: 0.0256
Epoch 11/20 - Train Loss: 0.0585 - Val Loss: 0.0164
Epoch 12/20 - Train Loss: 0.0527 - Val Loss: 0.0201
Epoch 13/20 - Train Loss: 0.0438 - Val Loss: 0.0113
Epoch 14/20 - Train Loss: 0.0386 - Val Loss: 0.0089
Epoch 15/20 - Train Loss: 0.0324 - Val Loss: 0.0084
Epoch 16/20 - Train Loss: 0.0304 - Val Loss: 0.0074
Epoch 17/20 - Train Loss: 0.0270 - Val Loss: 0.0089
Epoch 18/20 - Train Loss: 0.0245 - Val Loss: 0.0049
Epoch 19/20 - Train Loss: 0.0222 - Val Loss: 0.0111
Epoch 20/20 - Train L

In [15]:
# After training is complete
def save_model(model, path='transformer_infix_postfix.pth'):
    """Save the model state to a file"""
    torch.save({
        'model_state_dict': model.state_dict(),
        'vocab': VOCAB,
        'max_len': MAX_LEN,
        'config': {
            'd_model': d_model,
            'num_heads': num_heads,
            'num_layers': num_layers,
            'd_ff': d_ff,
            'dropout': dropout
        }
    }, path)
    print(f"Model saved to {path}")
save_model(model, '/content/drive/MyDrive/transformer_model.pth')

Model saved to /content/drive/MyDrive/transformer_model.pth


In [16]:
# -------------------- Inference --------------------
def autoregressive_decode(model, encoder_input, max_length=MAX_LEN):
    model.eval()
    with torch.no_grad():
        # Encode the input
        # Ensure encoder_input has batch dimension
        if encoder_input.ndim == 1:
            encoder_input = encoder_input.unsqueeze(0)

        # Generate encoder mask for padding (for encoder self-attention and decoder cross-attention)
        encoder_mask = (encoder_input != PAD_ID).unsqueeze(1).unsqueeze(2) # Shape (batch_size, 1, 1, src_seq_len)

        encoder_embedded = model.encoder_embedding(encoder_input)
        encoder_output = model.positional_encoding(encoder_embedded)

        for enc_layer in model.encoder_layers:
            # Use the padding mask for encoder self-attention
            encoder_output = enc_layer(encoder_output, encoder_mask)

        # Initialize decoder input with SOS token
        decoder_input = torch.tensor([[SOS_ID]], dtype=torch.long) # Start with batch size 1

        output_sequence = []

        for _ in range(max_length):
            # Prepare decoder input
            decoder_embedded = model.decoder_embedding(decoder_input)
            decoder_output = model.positional_encoding(decoder_embedded)

            # Generate masks for decoder
            # Target padding mask (not strictly needed during autoregression if we manage sequence length)
            # tgt_pad_mask = (decoder_input != PAD_ID).unsqueeze(1).unsqueeze(2)

            # Target causal mask (lookahead mask)
            tgt_seq_length = decoder_input.size(1)
            # Create square causal mask (seq_len, seq_len)
            nopeak_mask = (1 - torch.triu(torch.ones(tgt_seq_length, tgt_seq_length), diagonal=1)).bool()
            # Add batch and head dimensions for broadcasting
            tgt_causal_mask = nopeak_mask.unsqueeze(0).unsqueeze(0) # Shape (1, 1, tgt_seq_len, tgt_seq_len)

            # Combine masks if needed (not strictly necessary for this simple case, just use causal mask)
            # tgt_mask = tgt_pad_mask & tgt_causal_mask # Not needed here as decoder_input grows

            # The mask for decoder self-attention should be the causal mask
            decoder_self_attn_mask = tgt_causal_mask

            # The mask for decoder cross-attention is the encoder padding mask
            decoder_cross_attn_mask = encoder_mask # Shape (batch_size, 1, 1, src_seq_len)

            # Decode
            # Pass the appropriate masks to the decoder layers
            dec_output = decoder_output
            for dec_layer in model.decoder_layers:
                # Use causal mask for self-attention, encoder padding mask for cross-attention
                dec_output = dec_layer(dec_output, encoder_output, decoder_cross_attn_mask, decoder_self_attn_mask)

            # Get next token logits (only for the last token generated)
            logits = model.fc(dec_output[:, -1, :])
            next_token = torch.argmax(logits, dim=-1) # Shape (batch_size,) -> (1,)

            # Break if EOS is generated
            if next_token.item() == EOS_ID:
                break

            # Add to output sequence and prepare next input
            output_sequence.append(next_token.item())
            # Append the new token to the decoder input sequence
            decoder_input = torch.cat([decoder_input, next_token.unsqueeze(0)], dim=1) # Append to seq_len dim

        return output_sequence

In [17]:
# -------------------- Evaluation --------------------
def prefix_accuracy_single(y_true, y_pred, id_to_token, eos_id=EOS_ID, verbose=False):
    t_str = decode_sequence(y_true, id_to_token).split(' EOS')[0]
    p_str = decode_sequence(y_pred, id_to_token).split(' EOS')[0]
    t_tokens = t_str.strip().split()
    p_tokens = p_str.strip().split()
    max_len = max(len(t_tokens), len(p_tokens))

    match_len = sum(x == y for x, y in zip(t_tokens, p_tokens))
    score = match_len / max_len if max_len>0 else 0

    if verbose:
        print("TARGET :", ' '.join(t_tokens))
        print("PREDICT:", ' '.join(p_tokens))
        print(f"PREFIX MATCH: {match_len}/{len(t_tokens)} → {score:.2f}")

    return score

def test(model, no=20, rounds=10):
    rscores = []
    for i in range(rounds):
        print(f"Round {i+1}/{rounds}")
        test_dataset = InfixPostfixDataset(no)
        scores = []
        for j in range(no):
            encoder_input = test_dataset[j]['encoder_input']
            y_true = test_dataset[j]['target'].numpy()

            generated = autoregressive_decode(model, encoder_input)
            scores.append(prefix_accuracy_single(y_true, generated, id_to_token))

        round_score = np.mean(scores)
        rscores.append(round_score)
        print(f"Round score: {round_score:.4f}")

    return np.mean(rscores), np.std(rscores)

In [18]:
def load_model(path='transformer_infix_postfix.pth'):
    """Load the model state from a file"""
    checkpoint = torch.load(path)

    # Recreate the model architecture
    loaded_model = Transformer(
        len(checkpoint['vocab']),  # src_vocab_size
        len(checkpoint['vocab']),  # tgt_vocab_size
        checkpoint['config']['d_model'],
        checkpoint['config']['num_heads'],
        checkpoint['config']['num_layers'],
        checkpoint['config']['d_ff'],
        checkpoint['max_len'],
        checkpoint['config']['dropout']
    )

    # Load the trained weights
    loaded_model.load_state_dict(checkpoint['model_state_dict'])

    # Update global variables
    global VOCAB, token_to_id, id_to_token, VOCAB_SIZE, PAD_ID, EOS_ID, SOS_ID
    VOCAB = checkpoint['vocab']
    token_to_id = {tok: i for i, tok in enumerate(VOCAB)}
    id_to_token = {i: tok for tok, i in token_to_id.items()}
    VOCAB_SIZE = len(VOCAB)
    PAD_ID = token_to_id['PAD']
    EOS_ID = token_to_id['EOS']
    SOS_ID = token_to_id['SOS']

    print(f"Model loaded from {path}")
    return loaded_model

model = load_model('/content/drive/MyDrive/transformer_model.pth')

Model loaded from /content/drive/MyDrive/transformer_model.pth


In [24]:
# Run evaluation
mean_score, std_score = test(model)
print(f"\nFinal Evaluation - Mean Prefix Accuracy: {mean_score:.4f}, Std: {std_score:.4f}")

Round 1/10
Round score: 1.0000
Round 2/10
Round score: 1.0000
Round 3/10
Round score: 1.0000
Round 4/10
Round score: 1.0000
Round 5/10
Round score: 1.0000
Round 6/10
Round score: 1.0000
Round 7/10
Round score: 1.0000
Round 8/10
Round score: 1.0000
Round 9/10
Round score: 1.0000
Round 10/10
Round score: 1.0000

Final Evaluation - Mean Prefix Accuracy: 1.0000, Std: 0.0000


In [20]:
# Example translation
sample_idx = np.random.randint(len(val_dataset))
encoder_input = val_dataset[sample_idx]['encoder_input']
target = val_dataset[sample_idx]['target'].numpy()

generated = autoregressive_decode(model, encoder_input)

print("\nExample Translation:")
print("Infix:", decode_sequence(encoder_input.numpy(), id_to_token))
print("Target Postfix:", decode_sequence(target, id_to_token))
print("Generated Postfix:", decode_sequence(generated, id_to_token))
prefix_accuracy_single(target, generated, id_to_token, verbose=True)


Example Translation:
Infix: a
Target Postfix: a
Generated Postfix: a
TARGET : a
PREDICT: a
PREFIX MATCH: 1/1 → 1.00


1.0